# Temario: Fresnel, película delgada y multicapa

Este notebook reescribe la física principal de `electro_sim` en un formato más cercano a una libreta de clase: funciones planas, comentarios físicos y las mismas cuatro gráficas para cada caso.

Casos incluidos:

1. Interfaz simple, sin película delgada.
2. Película delgada antirreflectante de cuarto de onda.
3. Multicapa DBR de Bragg.

La intención no es reemplazar la app, sino leer y modificar el cálculo en un entorno familiar para física.

## 1. Notación y convenciones

Trabajamos con medios lineales, isotrópicos y homogéneos, caracterizados por permitividad relativa `ε` y permeabilidad relativa `μ`.

- Índice complejo: `n = sqrt(ε μ)`.
- Componente tangencial conservada: `kx = n_inc sin(θ_i)`.
- Componente normal: `kz = sqrt(n² - kx²)`.
- Admitancia TE: `q_TE = kz / μ`.
- Admitancia TM: `q_TM = kz / ε`.
- Coeficientes de amplitud: `r = (q1 - q2)/(q1 + q2)`, `t = 2q1/(q1 + q2)`.
- Reflectancia: `R = |r|²`.
- Transmitancia de potencia: `T = Re(q_out)/Re(q_in) |t|²`.
- Absorptancia: `A = 1 - R - T`.

Importante: aquí `A` es absorptancia, o fracción de potencia absorbida. No es la absorbancia espectroscópica `-log10(T)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.scimath import sqrt as csqrt

np.set_printoptions(precision=5, suppress=True)

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': True,
})

## 2. Funciones base

Estas funciones son deliberadamente pequeñas. Cada una representa una idea física: crear un medio, conservar `kx`, obtener `kz`, calcular admitancias y convertir amplitudes en potencias.

In [ ]:
FLUX_EPSILON = 1e-12


def medio_por_eps(eps, mu=1.0, nombre=''):
    """Crea un medio óptico a partir de ε y μ relativos."""
    eps = complex(eps)
    mu = complex(mu)
    return {
        'eps': eps,
        'mu': mu,
        'n': csqrt(eps * mu),
        'nombre': nombre,
    }


def medio_por_indice(n, mu=1.0, nombre=''):
    """Crea un medio no magnético usando n. Para μ=1, ε=n²."""
    n = complex(n)
    mu = complex(mu)
    eps = n**2 / mu
    return medio_por_eps(eps, mu=mu, nombre=nombre)


def capa_por_eps(eps, espesor_nm, mu=1.0, nombre=''):
    """Crea una capa finita. Es un medio más su espesor físico d."""
    capa = medio_por_eps(eps, mu=mu, nombre=nombre)
    capa['d_nm'] = float(espesor_nm)
    return capa


def capa_por_indice(n, espesor_nm, mu=1.0, nombre=''):
    """Crea una capa finita usando n y d."""
    n = complex(n)
    mu = complex(mu)
    eps = n**2 / mu
    return capa_por_eps(eps, espesor_nm, mu=mu, nombre=nombre)


def kx_desde_angulo(theta_rad, medio_incidente):
    """Componente tangencial conservada en todas las interfaces planas."""
    return medio_incidente['n'] * np.sin(theta_rad).astype(complex)


def kz_desde_kx(medio, kx):
    """Componente normal. La raíz compleja permite TIR y absorción."""
    return csqrt(medio['n']**2 - kx**2)


def fase_en_capa(kz, espesor_nm, lambda_nm):
    """Fase compleja acumulada al cruzar una capa de espesor d."""
    return (2 * np.pi * espesor_nm / lambda_nm) * kz


def admitancia(medio, kz, polarizacion):
    """Admitancia óptica para TE o TM."""
    if polarizacion == 'TE':
        return kz / medio['mu']
    if polarizacion == 'TM':
        return kz / medio['eps']
    raise ValueError("polarizacion debe ser 'TE' o 'TM'")


def coeficientes_interfaz(q_izq, q_der):
    """Coeficientes complejos de Fresnel en una sola frontera."""
    denominador = q_izq + q_der
    r = (q_izq - q_der) / denominador
    t = 2 * q_izq / denominador
    return r, t


def transmitancia_potencia(q_in, q_out, t):
    """Convierte amplitud transmitida en flujo de potencia normalizado."""
    flujo_in = np.real(q_in)
    flujo_out = np.real(q_out)
    flujo_seguro = np.where(np.abs(flujo_in) < FLUX_EPSILON, 1.0, flujo_in)
    T = np.where(
        np.abs(flujo_in) < FLUX_EPSILON,
        0.0,
        (flujo_out / flujo_seguro) * np.abs(t)**2,
    )
    return np.maximum(0.0, np.real(T))


def canal_optico(r, t, q_in, q_out):
    """Agrupa amplitudes, potencias y fases para una polarización."""
    R = np.abs(r)**2
    T = transmitancia_potencia(q_in, q_out, t)
    A = np.maximum(0.0, np.real(1.0 - R - T))
    return {
        'r': r,
        't': t,
        'R': R,
        'T': T,
        'A': A,
        'phi_r': np.angle(r, deg=True),
        'phi_t': np.angle(t, deg=True),
    }


def agregar_no_polarizada(resultado):
    """Promedia TE y TM para luz no polarizada."""
    resultado['unpolarized'] = {
        'R': 0.5 * (resultado['TE']['R'] + resultado['TM']['R']),
        'T': 0.5 * (resultado['TE']['T'] + resultado['TM']['T']),
        'A': 0.5 * (resultado['TE']['A'] + resultado['TM']['A']),
    }
    return resultado

## 3. Solucionadores planos

A partir de las funciones base, resolvemos tres geometrías: interfaz simple, película delgada y multicapa por matriz de transferencia. No se usan clases; cada función recibe arreglos y diccionarios físicos explícitos.

In [ ]:
def resolver_interfaz(theta_deg, medio_incidente, medio_sustrato):
    """Interfaz única: medio incidente -> sustrato semi-infinito."""
    theta_rad = np.radians(np.asarray(theta_deg, dtype=float))
    kx = kx_desde_angulo(theta_rad, medio_incidente)
    kz_1 = kz_desde_kx(medio_incidente, kx)
    kz_2 = kz_desde_kx(medio_sustrato, kx)

    resultado = {'theta_deg': np.asarray(theta_deg, dtype=float)}
    for pol in ('TE', 'TM'):
        q_1 = admitancia(medio_incidente, kz_1, pol)
        q_2 = admitancia(medio_sustrato, kz_2, pol)
        r, t = coeficientes_interfaz(q_1, q_2)
        resultado[pol] = canal_optico(r, t, q_1, q_2)

    return agregar_no_polarizada(resultado)


def resolver_pelicula_delgada(theta_deg, medio_incidente, pelicula, medio_sustrato, lambda_nm):
    """Película finita entre dos medios semi-infinitos, usando fórmula de Airy."""
    theta_rad = np.radians(np.asarray(theta_deg, dtype=float))
    kx = kx_desde_angulo(theta_rad, medio_incidente)

    kz_1 = kz_desde_kx(medio_incidente, kx)
    kz_f = kz_desde_kx(pelicula, kx)
    kz_2 = kz_desde_kx(medio_sustrato, kx)
    beta = fase_en_capa(kz_f, pelicula['d_nm'], lambda_nm)
    fase_ida_vuelta = np.exp(2j * beta)

    resultado = {'theta_deg': np.asarray(theta_deg, dtype=float)}
    for pol in ('TE', 'TM'):
        q_1 = admitancia(medio_incidente, kz_1, pol)
        q_f = admitancia(pelicula, kz_f, pol)
        q_2 = admitancia(medio_sustrato, kz_2, pol)

        r_01, t_01 = coeficientes_interfaz(q_1, q_f)
        r_12, t_12 = coeficientes_interfaz(q_f, q_2)

        denominador = 1 + r_01 * r_12 * fase_ida_vuelta
        r = (r_01 + r_12 * fase_ida_vuelta) / denominador
        t = (t_01 * t_12 * np.exp(1j * beta)) / denominador
        resultado[pol] = canal_optico(r, t, q_1, q_2)

    return agregar_no_polarizada(resultado)


def amplitudes_tmm(theta_deg, medio_incidente, capas, medio_sustrato, lambda_nm, polarizacion):
    """Amplitudes r y t de una pila multicapa por Transfer Matrix Method."""
    theta_rad = np.radians(np.asarray(theta_deg, dtype=float))
    kx = kx_desde_angulo(theta_rad, medio_incidente)
    n_angulos = kx.size

    kz_in = kz_desde_kx(medio_incidente, kx)
    kz_out = kz_desde_kx(medio_sustrato, kx)
    q_in = admitancia(medio_incidente, kz_in, polarizacion)
    q_out = admitancia(medio_sustrato, kz_out, polarizacion)

    M = np.zeros((2, 2, n_angulos), dtype=complex)
    M[0, 0] = 1.0
    M[1, 1] = 1.0

    for capa in capas:
        kz_capa = kz_desde_kx(capa, kx)
        q_capa = admitancia(capa, kz_capa, polarizacion)
        delta = fase_en_capa(kz_capa, capa['d_nm'], lambda_nm)

        cos_delta = np.cos(delta)
        sin_delta = np.sin(delta)

        M_capa = np.empty((2, 2, n_angulos), dtype=complex)
        M_capa[0, 0] = cos_delta
        M_capa[0, 1] = -1j * sin_delta / q_capa
        M_capa[1, 0] = -1j * q_capa * sin_delta
        M_capa[1, 1] = cos_delta

        # Producto matricial 2x2 para todos los ángulos a la vez.
        M = np.einsum('ijn,jkn->ikn', M, M_capa)

    denominador = q_in * M[0, 0] + q_in * q_out * M[0, 1] + M[1, 0] + q_out * M[1, 1]
    r = (q_in * M[0, 0] + q_in * q_out * M[0, 1] - M[1, 0] - q_out * M[1, 1]) / denominador
    t = 2 * q_in / denominador
    return r, t, q_in, q_out


def resolver_multicapa_tmm(theta_deg, medio_incidente, capas, medio_sustrato, lambda_nm):
    """Pila arbitraria de capas planas usando TMM."""
    resultado = {'theta_deg': np.asarray(theta_deg, dtype=float)}
    for pol in ('TE', 'TM'):
        r, t, q_in, q_out = amplitudes_tmm(
            theta_deg, medio_incidente, capas, medio_sustrato, lambda_nm, pol
        )
        resultado[pol] = canal_optico(r, t, q_in, q_out)

    return agregar_no_polarizada(resultado)


def construir_dbr(n_alto=2.3, n_bajo=1.45, pares=4, lambda_diseno_nm=550.0):
    """Espejo de Bragg: pares alternados de capas λ/4 alto-bajo índice."""
    d_alto = lambda_diseno_nm / (4 * n_alto)
    d_bajo = lambda_diseno_nm / (4 * n_bajo)

    capas = []
    for i in range(int(pares)):
        capas.append(capa_por_indice(n_alto, d_alto, nombre=f'H{i + 1}'))
        capas.append(capa_por_indice(n_bajo, d_bajo, nombre=f'L{i + 1}'))
    return capas

## 4. Función común de graficación

Para comparar casos sin cambiar la lectura visual, todos usan los mismos cuatro paneles: potencias `R/T`, absorptancia `A`, magnitudes de amplitud y fases.

In [ ]:
def _limite_superior_positivo(*arrays, minimo=0.05, margen=1.08):
    maximo = max(float(np.nanmax(np.real(a))) for a in arrays)
    return max(minimo, margen * maximo)


def graficar_cuatro_paneles(resultado, titulo, marcadores=None):
    """Dibuja las cuatro gráficas estándar para un resultado angular."""
    theta = resultado['theta_deg']
    marcadores = marcadores or []

    fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
    ax_rt, ax_a, ax_amp, ax_phase = axs.ravel()

    colores = {'TE': 'tab:blue', 'TM': 'tab:orange', 'unpolarized': 'black'}
    etiquetas = {'TE': 'TE', 'TM': 'TM', 'unpolarized': 'No pol.'}

    for pol in ('TE', 'TM', 'unpolarized'):
        ax_rt.plot(theta, resultado[pol]['R'], color=colores[pol], lw=1.8, label=f'R {etiquetas[pol]}')
        ax_rt.plot(theta, resultado[pol]['T'], color=colores[pol], lw=1.5, ls='--', label=f'T {etiquetas[pol]}')
    ax_rt.set_title('Reflectancia y transmitancia')
    ax_rt.set_ylabel('Fracción de potencia')
    ax_rt.set_ylim(0, 1.05)
    ax_rt.legend(ncol=2, fontsize=8)

    for pol in ('TE', 'TM', 'unpolarized'):
        ax_a.plot(theta, resultado[pol]['A'], color=colores[pol], lw=1.8, label=f'A {etiquetas[pol]}')
    ax_a.set_title('Absorptancia A = 1 - R - T')
    ax_a.set_ylabel('Fracción absorbida')
    ax_a.set_ylim(0, _limite_superior_positivo(resultado['TE']['A'], resultado['TM']['A'], resultado['unpolarized']['A']))
    ax_a.legend(fontsize=8)

    ax_amp.plot(theta, np.abs(resultado['TE']['r']), color='tab:blue', lw=1.8, label='|r| TE')
    ax_amp.plot(theta, np.abs(resultado['TM']['r']), color='tab:orange', lw=1.8, label='|r| TM')
    ax_amp.plot(theta, np.abs(resultado['TE']['t']), color='tab:blue', lw=1.5, ls='--', label='|t| TE')
    ax_amp.plot(theta, np.abs(resultado['TM']['t']), color='tab:orange', lw=1.5, ls='--', label='|t| TM')
    ax_amp.set_title('Magnitud de coeficientes')
    ax_amp.set_ylabel('Magnitud')
    ax_amp.set_ylim(0, _limite_superior_positivo(
        np.abs(resultado['TE']['r']), np.abs(resultado['TM']['r']),
        np.abs(resultado['TE']['t']), np.abs(resultado['TM']['t']),
        minimo=1.05,
    ))
    ax_amp.legend(fontsize=8)

    ax_phase.plot(theta, resultado['TE']['phi_r'], color='tab:blue', lw=1.8, label='φ_r TE')
    ax_phase.plot(theta, resultado['TM']['phi_r'], color='tab:orange', lw=1.8, label='φ_r TM')
    ax_phase.plot(theta, resultado['TE']['phi_t'], color='tab:blue', lw=1.5, ls='--', label='φ_t TE')
    ax_phase.plot(theta, resultado['TM']['phi_t'], color='tab:orange', lw=1.5, ls='--', label='φ_t TM')
    ax_phase.set_title('Fase de coeficientes')
    ax_phase.set_ylabel('Fase (grados)')
    ax_phase.set_ylim(-185, 185)
    ax_phase.legend(fontsize=8)

    for ax in axs.ravel():
        for x, texto in marcadores:
            ax.axvline(x, color='0.35', ls=':', lw=1)
        ax.set_xlim(theta[0], theta[-1])
        ax.set_xlabel('θᵢ (grados)')

    if marcadores:
        for x, texto in marcadores:
            ax_rt.text(x, 1.02, texto, rotation=90, va='top', ha='right', fontsize=8, color='0.25')

    fig.suptitle(titulo, fontsize=14)
    fig.tight_layout()
    return fig, axs

## 5. Parámetros comunes

Usaremos aire como medio incidente, vidrio como sustrato, `λ0 = 550 nm` y un barrido angular de `0°` a `89.9°`. Evitamos exactamente `90°` porque el flujo normal incidente se anula en incidencia rasante.

In [ ]:
lambda0_nm = 550.0
theta = np.linspace(0.0, 89.9, 600)

aire = medio_por_indice(1.0, nombre='Aire')
vidrio = medio_por_indice(1.5, nombre='Vidrio n=1.5')

print(f"lambda0 = {lambda0_nm:.1f} nm")
print(f"n_aire = {aire['n']:.3g}")
print(f"n_vidrio = {vidrio['n']:.3g}")

## 6. Caso 1: interfaz simple, sin película

Aquí solo existen dos medios semi-infinitos. La curva TM debe mostrar el mínimo de Brewster, donde la reflexión de polarización p se anula para medios no absorbentes y `μ = 1`.

In [ ]:
resultado_interfaz = resolver_interfaz(theta, aire, vidrio)
theta_brewster = np.degrees(np.arctan(vidrio['n'].real / aire['n'].real))

graficar_cuatro_paneles(
    resultado_interfaz,
    'Caso 1: interfaz simple aire → vidrio',
    marcadores=[(theta_brewster, 'Brewster TM')],
);

print(f"Ángulo de Brewster esperado: {theta_brewster:.2f}°")
print(f"R_TE(0°) = {resultado_interfaz['TE']['R'][0]:.4f}")
print(f"T_TE(0°) = {resultado_interfaz['TE']['T'][0]:.4f}")

## 7. Caso 2: película delgada λ/4

Para una capa antirreflectante ideal a incidencia normal:

- `n_f = sqrt(n_aire n_vidrio)`.
- `d = λ0/(4 n_f)`.

La cancelación ocurre porque los haces reflejados en ambas fronteras salen con desfase relativo destructivo en la longitud de onda de diseño.

In [ ]:
n_ar = np.sqrt(aire['n'].real * vidrio['n'].real)
d_ar_nm = lambda0_nm / (4 * n_ar)
pelicula_ar = capa_por_indice(n_ar, d_ar_nm, nombre='AR λ/4')

resultado_pelicula = resolver_pelicula_delgada(theta, aire, pelicula_ar, vidrio, lambda0_nm)

graficar_cuatro_paneles(
    resultado_pelicula,
    'Caso 2: película antirreflectante λ/4',
    marcadores=[(theta_brewster, 'Brewster base')],
);

print(f"n_AR = {n_ar:.5f}")
print(f"d_AR = {d_ar_nm:.2f} nm")
print(f"R_unpol(0°) = {resultado_pelicula['unpolarized']['R'][0]:.3e}")

## 8. Caso 3: multicapa DBR de Bragg

Un DBR alterna capas de índice alto y bajo con espesor óptico de cuarto de onda. En la longitud de diseño, las reflexiones parciales vuelven casi en fase hacia el medio incidente y aumentan mucho la reflectancia.

Usaremos `nH = 2.3`, `nL = 1.45`, `N = 4` pares y `λ0 = 550 nm`.

In [ ]:
nH = 2.3
nL = 1.45
pares = 4
capas_dbr = construir_dbr(nH, nL, pares, lambda0_nm)

resultado_dbr = resolver_multicapa_tmm(theta, aire, capas_dbr, vidrio, lambda0_nm)

graficar_cuatro_paneles(
    resultado_dbr,
    'Caso 3: multicapa DBR de Bragg',
);

espesor_total = sum(capa['d_nm'] for capa in capas_dbr)
print(f"Capas: {len(capas_dbr)} ({pares} pares H/L)")
print(f"d_H = {capas_dbr[0]['d_nm']:.2f} nm, d_L = {capas_dbr[1]['d_nm']:.2f} nm")
print(f"Espesor total = {espesor_total:.2f} nm")
print(f"R_unpol(0°) = {resultado_dbr['unpolarized']['R'][0]:.4f}")

## 9. Comparación rápida

La misma interfaz puede pasar de una reflexión moderada, a una cancelación en incidencia normal, a una alta reflexión por interferencia multicapa. Todas son consecuencias de sumar amplitudes complejas y luego convertir a potencia.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(theta, resultado_interfaz['unpolarized']['R'], lw=2, label='Interfaz simple')
ax.plot(theta, resultado_pelicula['unpolarized']['R'], lw=2, label='Película λ/4')
ax.plot(theta, resultado_dbr['unpolarized']['R'], lw=2, label='DBR 4 pares')
ax.set_title('Comparación de reflectancia no polarizada')
ax.set_xlabel('θᵢ (grados)')
ax.set_ylabel('R no polarizada')
ax.set_xlim(theta[0], theta[-1])
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout();

## 10. Validaciones físicas

Estas pruebas no hacen el notebook más bonito, pero sí lo vuelven confiable. Si alguna falla, hay una inconsistencia de signos, normalización de potencia o matriz de transferencia.

In [ ]:
def verificar(condicion, mensaje):
    """Levanta AssertionError aunque Python se ejecute con -O."""
    if not bool(condicion):
        raise AssertionError(mensaje)


def verificar_cercano(valor, esperado, atol, mensaje):
    """Comprueba cercanía escalar con un mensaje físico legible."""
    if not np.isclose(valor, esperado, atol=atol):
        raise AssertionError(
            f'{mensaje}: obtenido {valor:.6g}, esperado {esperado:.6g}, tolerancia {atol:.1e}'
        )


def verificar_arreglos_cercanos(actual, esperado, atol, mensaje):
    """Comprueba cercanía entre curvas completas."""
    if not np.allclose(actual, esperado, atol=atol):
        error_max = np.max(np.abs(actual - esperado))
        raise AssertionError(
            f'{mensaje}: error máximo {error_max:.3e}, tolerancia {atol:.1e}'
        )


def verificar_conservacion(resultado, atol=1e-8):
    for pol in ('TE', 'TM', 'unpolarized'):
        total = resultado[pol]['R'] + resultado[pol]['T'] + resultado[pol]['A']
        verificar_arreglos_cercanos(total, 1.0, atol, f'Falla conservación de energía en {pol}')


def run_physics_checks():
    lambda_test_nm = 550.0
    theta_test = np.linspace(0.0, 89.9, 800)
    aire_test = medio_por_indice(1.0, nombre='Aire')
    vidrio_test = medio_por_indice(1.5, nombre='Vidrio')

    simple = resolver_interfaz(theta_test, aire_test, vidrio_test)
    verificar_cercano(simple['TE']['R'][0], 0.04, 5e-4, 'R_TE(0°) aire-vidrio')
    verificar_cercano(simple['TM']['R'][0], 0.04, 5e-4, 'R_TM(0°) aire-vidrio')
    verificar_cercano(simple['TE']['T'][0], 0.96, 5e-4, 'T_TE(0°) aire-vidrio')

    theta_b_num = theta_test[np.argmin(simple['TM']['R'])]
    verificar(abs(theta_b_num - 56.31) < 0.2, f'Ángulo de Brewster fuera de tolerancia: {theta_b_num:.3f}°')
    verificar_conservacion(simple)

    # Una película de espesor cero debe colapsar a la interfaz directa.
    pelicula_cero = capa_por_eps(1.9, 0.0, nombre='capa d=0')
    cero = resolver_pelicula_delgada(theta_test, aire_test, pelicula_cero, vidrio_test, lambda_test_nm)
    for pol in ('TE', 'TM'):
        verificar_arreglos_cercanos(cero[pol]['R'], simple[pol]['R'], 1e-10, f'Película d=0 R {pol}')
        verificar_arreglos_cercanos(cero[pol]['T'], simple[pol]['T'], 1e-10, f'Película d=0 T {pol}')

    # Una sola capa por TMM debe coincidir con la fórmula cerrada de película delgada.
    pelicula = capa_por_eps(1.9, 100.0, nombre='película de prueba')
    film = resolver_pelicula_delgada(theta_test, aire_test, pelicula, vidrio_test, lambda_test_nm)
    tmm_una = resolver_multicapa_tmm(theta_test, aire_test, [pelicula], vidrio_test, lambda_test_nm)
    for pol in ('TE', 'TM', 'unpolarized'):
        verificar_arreglos_cercanos(tmm_una[pol]['R'], film[pol]['R'], 1e-10, f'TMM una capa R {pol}')
        verificar_arreglos_cercanos(tmm_una[pol]['T'], film[pol]['T'], 1e-10, f'TMM una capa T {pol}')
        verificar_arreglos_cercanos(tmm_una[pol]['A'], film[pol]['A'], 1e-10, f'TMM una capa A {pol}')
    verificar_conservacion(film)
    verificar_conservacion(tmm_una)

    # DBR lossless: conserva energía y refleja fuertemente cerca de incidencia normal.
    dbr = resolver_multicapa_tmm(
        theta_test,
        aire_test,
        construir_dbr(2.3, 1.45, 4, lambda_test_nm),
        vidrio_test,
        lambda_test_nm,
    )
    verificar_conservacion(dbr)
    verificar(dbr['TE']['R'][0] > 0.85, f"DBR TE refleja poco en 0°: {dbr['TE']['R'][0]:.4f}")
    verificar(dbr['TM']['R'][0] > 0.85, f"DBR TM refleja poco en 0°: {dbr['TM']['R'][0]:.4f}")

    print('Todas las validaciones físicas pasaron.')


run_physics_checks()

## 11. Cargar una multicapa desde CSV

Esta celda reproduce en el notebook la nueva entrada de la app: selecciona un CSV con el explorador de archivos, carga una pila arbitraria de capas y verifica conservaci?n de energ?a. Si cancelas el di?logo, usa el ejemplo `examples/multilayer_100_layers.csv` incluido en el repositorio.


In [ ]:
from pathlib import Path
import csv


def _repo_root_desde_notebook():
    """Busca la raiz del repo desde el directorio actual del notebook."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'examples' / 'multilayer_100_layers.csv').exists():
            return candidate
    return cwd


def seleccionar_csv_capas(default_path):
    """Abre explorador de archivos; si no hay GUI o se cancela, usa default_path."""
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        selected = filedialog.askopenfilename(
            title='Seleccionar CSV de capas',
            initialdir=str(default_path.parent),
            filetypes=[('CSV', '*.csv'), ('Todos los archivos', '*.*')],
        )
        root.destroy()
    except Exception as exc:
        print(f'No se pudo abrir el explorador de archivos ({exc}); usando ejemplo del repo.')
        selected = ''
    return Path(selected) if selected else default_path


def _normalizar_encabezados(fieldnames):
    if not fieldnames:
        raise ValueError('El CSV no tiene encabezados.')
    return {name.strip().lower(): name for name in fieldnames if name and name.strip()}


def _fila_vacia(row):
    return all(str(value or '').strip() == '' for value in row.values())


def _numero(row, headers, key, fila):
    raw = str(row.get(headers[key], '') or '').strip()
    if raw == '':
        raise ValueError(f'Fila {fila}, columna {key}: valor requerido.')
    return float(raw)


def cargar_capas_csv_notebook(csv_path):
    """Carga formato n_re/n_im o eps/mu en las estructuras del notebook."""
    capas = []
    with csv_path.open('r', newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        headers = _normalizar_encabezados(reader.fieldnames)
        por_indice = {'n_re', 'n_im', 'thickness_nm'} <= set(headers)
        por_eps_mu = {'eps_re', 'eps_im', 'mu_re', 'mu_im', 'thickness_nm'} <= set(headers)
        if not (por_indice or por_eps_mu):
            raise ValueError(
                'El CSV debe tener name,n_re,n_im,thickness_nm '
                'o eps_re,eps_im,mu_re,mu_im,thickness_nm.'
            )

        for fila, row in enumerate(reader, start=2):
            if _fila_vacia(row):
                continue
            nombre = str(row.get(headers.get('name', ''), '') or f'L{len(capas) + 1}').strip()
            d_nm = _numero(row, headers, 'thickness_nm', fila)
            if d_nm <= 0:
                raise ValueError(f'Fila {fila}: thickness_nm debe ser > 0.')

            if por_indice:
                n_re = _numero(row, headers, 'n_re', fila)
                n_im = _numero(row, headers, 'n_im', fila)
                if n_re <= 0 or n_im < 0:
                    raise ValueError(f'Fila {fila}: n_re debe ser > 0 y n_im >= 0.')
                capas.append(capa_por_indice(n_re + 1j * n_im, d_nm, nombre=nombre))
            else:
                eps = _numero(row, headers, 'eps_re', fila) + 1j * _numero(row, headers, 'eps_im', fila)
                mu = _numero(row, headers, 'mu_re', fila) + 1j * _numero(row, headers, 'mu_im', fila)
                capas.append(capa_por_eps(eps, d_nm, mu=mu, nombre=nombre))

    if not capas:
        raise ValueError('El CSV no contiene capas validas.')
    return capas


repo_root = _repo_root_desde_notebook()
csv_default = repo_root / 'examples' / 'multilayer_100_layers.csv'
csv_capas = seleccionar_csv_capas(csv_default)
capas_csv = cargar_capas_csv_notebook(csv_capas)
resultado_csv = resolver_multicapa_tmm(theta, aire, capas_csv, vidrio, lambda0_nm)

graficar_cuatro_paneles(resultado_csv, f'Multicapa desde CSV: {csv_capas.name}')

espesor_total = sum(capa['d_nm'] for capa in capas_csv)
print(f'Archivo: {csv_capas}')
print(f'Capas: {len(capas_csv)}')
print(f'Espesor total = {espesor_total:.3f} nm')
for pol in ('TE', 'TM', 'unpolarized'):
    total = resultado_csv[pol]['R'] + resultado_csv[pol]['T'] + resultado_csv[pol]['A']
    error = np.max(np.abs(total - 1.0))
    print(f'max |R+T+A-1| {pol}: {error:.3e}')

verificar_conservacion(resultado_csv, atol=1e-8)
print('CSV cargado y validado fisicamente.')
